# INTEGRATES Building-Energy Calibration Challenge

## 00 - Challenge contract

Your task is to select a subset of the twelve available calibration parameters and replace the search strategy in the repository-root `calibration.py`. The search space contains uncertain operating schedules, controls, and tightly bounded heat-loss multipliers; audited values remain the fixed defaults. A two-regime model represents normally heated space and wings/floors operated at lower temperatures. Bolzano's legal heating season and dated exceptions are fixed challenge inputs, not calibration variables. Do not change the challenge-owned simulation, inputs, measurements, metrics, or export code.

Your file must define:

- `TEAM_NAME: str`
- `ALGORITHM_NAME: str`
- `calibrate(objective, parameter_space, budget, seed) -> dict[str, float]`

The returned dictionary must contain every supplied parameter name exactly once, with finite values inside the stated bounds. Call `objective(vector)` for a sequential evaluation. Methods that naturally propose a population may call `objective.evaluate_many(vectors)`; both interfaces use the same challenge-controlled local process pool, cache, validation, and evaluation budget. The vector order is the order of `parameter_space`. Every process runs on the laptop executing this notebook—there is no central simulation server or cross-laptop process pool. The official scored run uses buildings 01–03, years 2018–2019, January warm-up, and February/March scoring.


## 01 - Imports & Initialization

In [ ]:
import integrates_hackathon.notebook_workflow as workflow

# Official scoring configuration. Teams may temporarily shorten this list
# while developing, but final verification restores both official years.
CALIBRATION_YEARS = [2018] #[2018, 2019] # The challenge will be judged on February and March of 2018 and 2019; during development you could reduce to just one year for faster iterations.
EVALUATION_BUDGET = 24 #48  # The contest will be scored with budget of 48, but during development you could reduce this for faster iterations.
PARALLEL_WORKERS = 6  # Reduce this if the laptop has fewer available cores.

context = workflow.initialize_challenge(
    backend="pbe",
    # building_ids=("building_01", "building_02", "building_03"),
    building_ids=("building_02",), ## During development, you may wish to calibrate only one building for faster iterations --> recommend building_02
    calibration_years=tuple(CALIBRATION_YEARS),
    score_months=(2, 3),
    warmup_month=1,
    evaluation_budget=EVALUATION_BUDGET,
    parallel_workers=PARALLEL_WORKERS,
    seed=2026,
)
workflow.show_challenge_contract(context)


## 02 - Visualize Input Data

This section parses inputs for `building_01`, `building_02`, and `building_03` and displays the February/March bills and 2018–2019 weather. Fixed audited properties and the calibration bounds remain visible. Bolzano's legal heating season and dated exceptions are fixed inputs, not variables teams may change.


In [ ]:
input_views = workflow.show_input_data(context)

## 03 - Load Baseline Energy Simulation

The notebook loads organizer-precomputed exact-window baseline results. Each selected year simulates every hour of January, February, and March; January is thermal warm-up and is discarded. Only February and March are compared with bills, with no annualization. Challengers do not rerun the baseline.

In [ ]:
baseline = workflow.load_baseline(context)
baseline_results = workflow.show_baseline_results(context, baseline)

## 04 - Run Calibration

This section imports and runs the repository-root `calibration.py`. The shipped starter evaluates the default vector once and returns it unchanged, so its calibrated score exactly matches the baseline until participants replace the passthrough with a search strategy. Each laptop creates its own local worker pool; no simulations or data leave that computer. Independent weather years run in separate processes. Participant methods may use the optional `objective.evaluate_many(vectors)` interface when they can propose candidates together. Before each exact-window task, the parent process prints the building, scored month/year, and calibration-set counter.

The workflow passes `parameter_space`, `budget`, and `seed` into `calibration.py`. Each item in `parameter_space` provides `.name`, `.lower`, `.upper`, and `.default`; participant code should read these values rather than duplicate bounds or assume a particular budget. Teams may use larger local budgets while developing, but final verification runs their unchanged `calibration.py` in a clean challenge copy with the official budget, buildings, years, months, seed, and parameter contract. The objective stops with an error if code requests more than the permitted number of new candidates.

A submission returns one scalar value per active parameter and building. That same value is applied to every scored month and year. Keys containing special months/years, nested schedules, and extra return keys are rejected. Final verification transfers `calibration.py` plus the active-parameter selection recorded in the standardized JSON; edits to simulation, inputs, scoring, or notebook files do not accompany a submission.


In [ ]:
### Saved experiments (participant helper, not organizer code) ###
# load_or_run_calibration() replaces workflow.run_calibration() in the next cell.
# If outputs/runs/ already holds this experiment (same algorithm, budget, years,
# buildings, seed, active parameters, and unchanged code of the selected
# algorithm), the saved result is loaded instead of being rerun. Only the code
# the selected algorithm uses is compared (see algorithm_fingerprint), so editing
# another algorithm, comments, or docstrings does not force a rerun.
# Delete the run folder to force a rerun.
import ast
import copy
import hashlib
import json
import re
from pathlib import Path

import pandas as pd

from integrates_hackathon.models import MonthlyConsumption
from integrates_hackathon.notebook_workflow import CalibrationCaseRun, SubmissionRun
from integrates_hackathon.objective import ObjectiveEvaluation
from integrates_hackathon.submission_api import load_submission

RUNS_DIR = Path(context.output_dir) / "runs"
EXPERIMENT_FILE = "experiment.json"  # algorithm fingerprint, stored next to the export
calibration_run_loaded_from = None  # set to the run folder when a saved run is reused
calibration_run_fingerprint = None  # fingerprint of the code used for calibration_run


def experiment_dir(context, algorithm_name):
    """Folder in outputs/runs/ that identifies one experiment configuration."""
    name = "__".join(
        [
            algorithm_name,
            f"budget{context.evaluation_budget}",
            "+".join(str(year) for year in context.calibration_years),
            "+".join(case.building_id.replace("building_", "b") for case in context.cases),
        ]
    )
    return RUNS_DIR / re.sub(r"[^A-Za-z0-9_+.-]+", "_", name)


def _strip_docstrings(node):
    for item in ast.walk(node):
        if (
            isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
            and item.body
            and isinstance(item.body[0], ast.Expr)
            and isinstance(item.body[0].value, ast.Constant)
            and isinstance(item.body[0].value.value, str)
        ):
            item.body = item.body[1:] or [ast.Pass()]
    return node


def algorithm_fingerprint(source_path, algorithm_name):
    """Fingerprint only the code the selected algorithm actually runs.

    Starts at calibrate() and the function registered for algorithm_name, then
    follows every top-level function, class, constant, and import they use.
    A registry dict mapping names to functions (like _ALGORITHMS) is followed
    only for the selected entry, so editing or adding other algorithms does not
    invalidate this one. Comments, formatting, and docstrings are ignored.
    Returns (fingerprint, {top-level name: hash}).
    """
    tree = ast.parse(Path(source_path).read_text(encoding="utf-8"))
    top = {}
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            top[node.name] = node
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                for name in ast.walk(target):
                    if isinstance(name, ast.Name):
                        top[name.id] = node
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            for alias in node.names:
                top[(alias.asname or alias.name).split(".")[0]] = node
    functions = {n for n, node in top.items() if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))}

    def registry(node):
        value = getattr(node, "value", None) if isinstance(node, (ast.Assign, ast.AnnAssign)) else None
        if isinstance(value, ast.Dict) and value.values and all(
            isinstance(v, ast.Name) and v.id in functions for v in value.values
        ):
            return value
        return None

    roots = ["calibrate"]
    for node in {id(n): n for n in top.values()}.values():
        found = registry(node)
        if found is not None:
            roots += [v.id for k, v in zip(found.keys, found.values)
                      if isinstance(k, ast.Constant) and k.value == algorithm_name]
    seen, stack = set(), list(roots)
    while stack:
        name = stack.pop()
        if name in seen or name not in top:
            continue
        seen.add(name)
        if registry(top[name]) is not None:
            continue  # do not pull in every registered algorithm
        stack += [n.id for n in ast.walk(top[name]) if isinstance(n, ast.Name)]
    parts = {
        name: hashlib.sha256(ast.dump(_strip_docstrings(copy.deepcopy(top[name]))).encode()).hexdigest()[:16]
        for name in sorted(seen)
        if registry(top[name]) is None
    }
    parts["<selected>"] = hashlib.sha256(f"{algorithm_name}->{roots[1:]}".encode()).hexdigest()[:16]
    fingerprint = hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()
    return fingerprint, parts


def _same_settings(meta, context):
    return (
        meta.get("seed") == context.seed
        and meta.get("backend") == context.backend
        and meta.get("evaluation_budget_per_building") == context.evaluation_budget
        and meta.get("active_parameter_selection")
        == {spec.name: float(spec.default) for spec in context.cases[0].parameters}
        and sorted(r["building_id"] for r in meta.get("building_results", []))
        == sorted(case.building_id for case in context.cases)
    )


def _load_saved_run(folder, context, submission):
    read = lambda name: pd.read_csv(folder / name, encoding="utf-8-sig", float_precision="round_trip")
    meta = json.loads((folder / "submission_result.json").read_text(encoding="utf-8"))
    summary = read("summary.csv").set_index("building_id")
    monthly = read("monthly_results.csv")
    trace = read("evaluation_trace.csv")
    cases = {}
    for case in context.cases:
        b = case.building_id
        rows = monthly[monthly["building_id"] == b]
        steps = trace[trace["building_id"] == b].sort_values("evaluation_number")
        final = steps.loc[steps["is_final_candidate"], "evaluation_number"]
        cases[b] = CalibrationCaseRun(
            building_id=b,
            parameters=dict(meta["parameters"][b]),
            simulated=tuple(
                MonthlyConsumption(int(r.year), int(r.month), float(r.calibrated_kwh))
                for r in rows.itertuples()
            ),
            evaluations=int(summary.loc[b, "evaluations"]),
            evaluation_trace=tuple(
                ObjectiveEvaluation(
                    loss=float(r["loss_wape_pct"]),
                    parameters={name: float(r[name]) for name in case.parameter_names},
                    metrics=None,  # not stored in the export; not needed downstream
                    simulated=(),
                )
                for _, r in steps.iterrows()
            ),
            runtime_seconds=float(summary.loc[b, "runtime_seconds"]),
            status=str(summary.loc[b, "status"]),
            final_evaluation_number=int(final.iloc[0]) if len(final) else None,
        )
    return SubmissionRun(submission=submission, cases=cases)


def _code_changes(saved_parts, parts):
    names = sorted(set(saved_parts) | set(parts))
    return [n for n in names if saved_parts.get(n) != parts.get(n)]


def load_or_run_calibration(context):
    global calibration_run_loaded_from, calibration_run_fingerprint
    submission = load_submission()
    fingerprint, parts = algorithm_fingerprint(submission.source_path, submission.algorithm_name)
    calibration_run_fingerprint = {
        "algorithm_fingerprint": fingerprint,
        "fingerprint_parts": parts,
        "calibration_py_sha256": submission.source_sha256,
    }
    folder = experiment_dir(context, submission.algorithm_name)
    meta_path = folder / "submission_result.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        experiment_path = folder / EXPERIMENT_FILE
        if experiment_path.exists():
            saved = json.loads(experiment_path.read_text(encoding="utf-8"))
            same_code = saved.get("algorithm_fingerprint") == fingerprint
            changed = _code_changes(saved.get("fingerprint_parts", {}), parts)
        else:  # archived before fingerprints existed: fall back to the whole-file hash
            same_code = meta.get("calibration_py_sha256") == submission.source_sha256
            changed = ["calibration.py (old archive without fingerprint)"]
            if same_code:
                experiment_path.write_text(json.dumps(calibration_run_fingerprint, indent=2), encoding="utf-8")
        if same_code and _same_settings(meta, context):
            calibration_run_loaded_from = folder
            print(
                f"Experiment already saved; LOADED (not rerun) from {folder}\n"
                f"Saved overall score: {meta['overall_score']:.2f}. Delete the folder to force a rerun."
            )
            return _load_saved_run(folder, context, submission)
        reason = f"algorithm code changed: {', '.join(changed)}" if not same_code else "seed or active parameters changed"
        print(f"Saved run in {folder.name} differs ({reason}); rerunning.")
    calibration_run_loaded_from = None
    return workflow.run_calibration(context)


In [ ]:
### Calibratable parameters ###
# Replace None with an in-range numeric starting value to activate a parameter.
# None holds that parameter at the organizer default shown in its comment.
calibratable_params_init = {
    "weekday_heating_hours": 8, # active occupied-setpoint hours per weekday; default 8; simulator rounds to a whole hour ## allowable range [4,12]
    "non_workday_heating_hours": 2, # active hours on weekends/public holidays; default 2; simulator rounds to a whole hour ## allowable range [0,4]
    "weekday_start_hour": 7, # start of active weekday heating; default 7; simulator rounds to a whole hour ## allowable range [5,9]
    "occupied_heating_setpoint_c": 20.0, # normal-regime setpoint during active hours; default 20 ## allowable range [18.5,21.5]
    "night_setback_temperature_c": 15.0, # maintained setpoint outside active hours, with energy counted; default 15 ## allowable range [12,18]
    "outdoor_shutdown_temperature_c": 15.0, # 24-hour mean outdoor threshold switching active heating to setback; default 15 ## allowable range [12,18]
    "normally_heated_area_fraction": 0.8, # conditioned area following normal rather than reduced regime; default 0.8 ## allowable range [0.6,1.0]
    "reduced_area_setpoint_c": 15.0, # active setpoint for reduced-regime areas; default 15 ## allowable range [12,17]
    "reduced_area_schedule_fraction": 0.5, # reduced active hours divided by normal active hours; default 0.5 ## allowable range [0.25,0.75]
    "air_exchange_multiplier": 1.0, # multiplier on audited/default infiltration and ventilation; default 1.0 ## allowable range [0.7,1.3]
    "opaque_envelope_u_value_multiplier": 1.0, # common audited wall/roof/floor U-value multiplier; default 1.0 ## allowable range [0.9,1.1]
    "window_u_value_multiplier": 1.0, # audited window U-value multiplier; default 1.0 ## allowable range [0.9,1.1]
}

# Organizer-owned assertions stop the run if keys change or if any activated
# starting value is non-numeric or outside its organizer-owned range.
context = workflow.configure_parameter_contract(
    context, calibratable_params_init
)

calibration_run = load_or_run_calibration(context)  # loads a saved identical run instead of rerunning
calibrated_parameters = workflow.show_calibration_results(context, calibration_run)

## 05 - Validate & Compare Results

The organizer-owned workflow validates every returned value and compares the independently rerun calibrated simulation with the baseline and measured monthly consumption.

In [ ]:
comparison_results = workflow.show_calibrated_comparison(
    context, baseline, calibration_run
)

## 06 - Scoring

WAPE is calculated only from the February/March bills in `CALIBRATION_YEARS`; nothing is annualized. The example positive score is `max(0, 100 - WAPE%)`. Each building is weighted equally so a large building cannot dominate. Baseline score, improvement, CV(RMSE), NMBE, and evaluation count remain visible for interpretation and possible tie-breaking.

In [ ]:
score_report = workflow.score_and_visualize(context, baseline, calibration_run)

## 07 - Export

Every team produces the same four artifacts: a one-row-per-building summary, a one-row-per-building-month comparison, an organizer-recorded trace of every unique budget-charged evaluation, and a machine-readable JSON result containing metadata, scores, active parameter selection, the calibration.py SHA-256 hash, and calibrated parameters.

In [ ]:
exported_paths = workflow.export_standardized_results(context, calibration_run, score_report)
workflow.show_exported_results(exported_paths)

## 08 - Algorithm comparison (development only)

Section 07 always exports to `outputs/<team-name>/`, so every run overwrites the previous one. This section keeps a copy of each run and compares them from the exported files only; nothing is simulated here.

**Workflow**
1. Set `ALGORITHM_NAME` in `calibration.py`.
2. Run Sections 01–07, then **08a**, which archives the export to `outputs/runs/<algorithm>__budget<N>__<years>__<buildings>/`. If the same experiment is already archived, Section 04 loads it instead of rerunning and says so. "Same" means the same algorithm, configuration, seed and active parameters, and unchanged code for the selected algorithm. Only the code that algorithm actually uses is compared (`calibrate`, the algorithm's function, and the helpers, constants and imports they reference). Editing another algorithm, comments or docstrings doesn't force a rerun. If anything relevant changed, Section 04 names it, reruns, and overwrites the archive.
3. Repeat for each algorithm to compare. You only need to rerun Section 04 onward, because `calibration.py` is reloaded on every run.
4. Run **08b**, which loads every archived run (or the ones listed in `RUNS_TO_COMPARE`) and shows the tables and plots.

### 08a - Archive this run

In [ ]:
import shutil

archive_dir = experiment_dir(context, calibration_run.submission.algorithm_name)
if calibration_run_loaded_from is not None:
    print(f"Run was loaded from {calibration_run_loaded_from}; archive left unchanged.")
else:
    archive_dir.mkdir(parents=True, exist_ok=True)
    for exported in exported_paths.values():
        shutil.copy2(exported, archive_dir / Path(exported).name)
    (archive_dir / EXPERIMENT_FILE).write_text(json.dumps(calibration_run_fingerprint, indent=2), encoding="utf-8")
    print(f"Archived run to {archive_dir}")
print("Archived runs so far:", sorted(d.name for d in RUNS_DIR.iterdir() if d.is_dir()))


### 08b - Compare archived runs

Leave `RUNS_TO_COMPARE = None` to compare every folder in `outputs/runs/`, or list folder names to pick some. Only compare runs with the same buildings and years when judging scores. The run label shows both.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

RUNS_DIR = Path(context.output_dir) / "runs"
RUNS_TO_COMPARE = None  # e.g. ["random_search__budget12__2018__b02", "bayesian_optimization__budget12__2018__b02"]

def _read_csv(path):
    return pd.read_csv(path, encoding="utf-8-sig")  # exports are written with a BOM

runs = {}
for folder in sorted(d for d in RUNS_DIR.glob("*") if (d / "submission_result.json").exists()):
    if RUNS_TO_COMPARE is not None and folder.name not in RUNS_TO_COMPARE:
        continue
    runs[folder.name] = {
        "meta": json.loads((folder / "submission_result.json").read_text(encoding="utf-8")),
        "summary": _read_csv(folder / "summary.csv"),
        "monthly": _read_csv(folder / "monthly_results.csv"),
        "trace": _read_csv(folder / "evaluation_trace.csv"),
    }
if not runs:
    raise FileNotFoundError(f"No archived runs in {RUNS_DIR}. Run Sections 04-07 and 08a first.")
print(f"Loaded {len(runs)} run(s):", *runs, sep="\n  ")

# Scores: one column per run, one row per building, plus the official overall mean.
score_table = pd.concat(
    {name: run["summary"].set_index("building_id")["calibrated_score"] for name, run in runs.items()},
    axis=1,
)
score_table.loc["OVERALL"] = {name: run["meta"]["overall_score"] for name, run in runs.items()}
display(Markdown("### Calibrated score (higher is better)"))
display(score_table.style.format("{:.2f}", na_rep="-").highlight_max(axis=1, color="#c6efce"))

detail_rows = []
for name, run in runs.items():
    final = run["trace"][run["trace"]["is_final_candidate"]].set_index("building_id")["evaluation_number"]
    for row in run["summary"].itertuples():
        detail_rows.append(
            {
                "run": name,
                "building_id": row.building_id,
                "baseline_score": row.baseline_score,
                "calibrated_score": row.calibrated_score,
                "improvement_points": row.improvement_points,
                "wape_pct": row.calibrated_weighted_absolute_percentage_error,
                "nmbe_pct": row.calibrated_nmbe_pct,
                "cv_rmse_pct": row.calibrated_cv_rmse_pct,
                "evaluations": row.evaluations,
                "best_found_at_eval": final.get(row.building_id),
                "runtime_min": row.runtime_seconds / 60,
            }
        )
display(Markdown("### Details"))
display(pd.DataFrame(detail_rows).set_index(["building_id", "run"]).sort_index().round(2))

display(Markdown("### Best parameters found"))
display(
    pd.DataFrame(
        [
            {"building_id": building, "run": name, **values}
            for name, run in runs.items()
            for building, values in run["meta"]["parameters"].items()
        ]
    ).set_index(["building_id", "run"]).sort_index().round(3)
)


In [ ]:
buildings = sorted({b for run in runs.values() for b in run["summary"]["building_id"]})
width = max(8, 5 * len(buildings))

# One fixed color per experiment, shared by both plots. Colors follow the sorted
# list of all archived runs, so a run keeps its color when you filter RUNS_TO_COMPARE.
all_run_names = sorted(d.name for d in RUNS_DIR.iterdir() if (d / "submission_result.json").exists())
palette = plt.get_cmap("tab10").colors
run_colors = {name: palette[i % len(palette)] for i, name in enumerate(all_run_names)}
reference_colors = {"measured": "#222222", "baseline": "#b0b0b0"}

# Convergence: best WAPE found so far vs. number of budget-charged evaluations.
fig, axes = plt.subplots(1, len(buildings), figsize=(width, 4), squeeze=False)
for ax, building in zip(axes[0], buildings):
    baseline_wape = None
    for name, run in runs.items():
        trace = run["trace"][run["trace"]["building_id"] == building].sort_values("evaluation_number")
        if trace.empty:
            continue
        line, = ax.plot(trace["evaluation_number"], trace["loss_wape_pct"].cummin(), marker=".", label=name, color=run_colors[name])
        ax.scatter(trace["evaluation_number"], trace["loss_wape_pct"], s=8, alpha=0.25, color=line.get_color())
        summary = run["summary"].set_index("building_id")
        baseline_wape = summary.loc[building, "baseline_weighted_absolute_percentage_error"]
    if baseline_wape is not None:
        ax.axhline(baseline_wape, color=reference_colors["baseline"], linestyle="--", linewidth=1, label="baseline")
    ax.set(title=building, xlabel="evaluation #", ylabel="WAPE % (lower is better)", yscale="log")
    ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=8)
fig.suptitle("Convergence: best-so-far WAPE (line) and every candidate (dots)")
fig.tight_layout()
plt.show()

# Monthly energy: measured vs. baseline vs. each run's calibrated result.
fig, axes = plt.subplots(1, len(buildings), figsize=(width, 4), squeeze=False)
for ax, building in zip(axes[0], buildings):
    frame = None
    for name, run in runs.items():
        monthly = run["monthly"][run["monthly"]["building_id"] == building].copy()
        if monthly.empty:
            continue
        monthly["period"] = monthly["year"].astype(str) + "-" + monthly["month"].map("{:02d}".format)
        monthly = monthly.set_index("period")
        if frame is None:
            frame = monthly[["actual_kwh", "baseline_kwh"]].rename(
                columns={"actual_kwh": "measured", "baseline_kwh": "baseline"}
            )
        frame[name] = monthly["calibrated_kwh"]
    colors = [reference_colors.get(column) or run_colors[column] for column in frame.columns]
    (frame / 1000).plot.bar(ax=ax, rot=0, width=0.8, color=colors)
    ax.set(title=building, ylabel="MWh", xlabel="")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Monthly heating energy: measured vs. simulated")
fig.tight_layout()
plt.show()